## Survey Existing Research

### Core Architectural Paradigms
    Single-Agent Systems: A single model instance with access to memory, defined APIs, and tools. It plans, acts, reflects, and repeats until the goal is reached. Best for focused, low-complexity use cases.
    Multi-Agent Systems: Multiple agents coordinate to solve complex problems.
    Hierarchical (Vertical): A leader agent acts as a supervisor, breaking down goals and delegating subtasks to specialist agents. Excellent for auditability and clear accountability.
    Horizontal (Decentralized): Agents act as peers, freely collaborating, sharing resources, and coordinating dynamically to solve interdisciplinary problems.
    Sequential: Agents operate in a linear chain where the output of one agent serves as the input for the next (often using routers to hand off work).

### The main orchestration platforms include the following:
    LangChain: Provides primitives for integrating tools, memory banks, and chains.
    LangGraph: Models agent workflows as stateful graphs, which is highly effective for programming loops, branches, and complex multi-agent flows.
    CrewAI: Organizes agents around a role-based, human-like division of labor, allowing for clear goal-setting and process tracking.
    AutoGen: An open-source multi-agent framework designed to let different AI agents converse and collaborate programmatically.
    Deep Agents: A specialized category of AI agents designed for complex, long-running tasks over extended time horizons. Unlike traditional "shallow" agents that simply loop between thinking and acting, deep agents can reason, plan, and autonomously manage themselves.

### Documented available code examples/notebooks
   1. Building-an-agent-with-langgraph
      https://www.kaggle.com/code/markishere/day-3-building-an-agent-with-langgraph
   2. Agent Sessions
      https://www.kaggle.com/code/kaggle5daysofai/day-3a-agent-sessions#%F0%9F%9A%80-Memory-Management---Part-1---Sessions
   3. Customize Deep Agents
      https://docs.langchain.com/oss/python/deepagents/customization
   4. A practical guide to building agents
      https://openai.com/business/guides-and-resources/a-practical-guide-to-building-ai-agents/
   5. Agentic AI architecture 101: An enterprise guide
      https://akka.io/blog/agentic-ai-architecture
   6. LangChain's Deep Agents: A Guide With Demo Project
      https://www.datacamp.com/tutorial/deep-agents

## Customize Deep Agents Based on Reproducing Available Solutions 

### 1. Set Up

1.1 Install dependencies

In [3]:
import sys
print(sys.executable)

C:\Users\myliu\anaconda3\python.exe


!{sys.executable} -m pip install deepagents

!pip install pyarrow

In [6]:
import deepagents
print(deepagents.__file__)

C:\Users\myliu\anaconda3\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.2 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "
C:\Users\myliu\anaconda3\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.2 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "
C:\Users\myliu\anaconda3\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.2 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "
C:\Users\myliu\anaconda3\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.2 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "
C:\Users\myl

C:\Users\myliu\anaconda3\Lib\site-packages\deepagents\__init__.py


In [7]:
!pip install python-dotenv

1.2 Configure API Key

In [9]:
import os
from dotenv import load_dotenv

try:
    # Load variables from .env
    load_dotenv()

    # Retrieve API key
    google_api_key = os.getenv("GOOGLE_API_KEY")

    MODEL_NAME = "gemini-3.5-flash"
    
    MAX_ITERATIONS = 10

    if not google_api_key:
        raise ValueError("GOOGLE_API_KEY not found in .env file")

    os.environ["GOOGLE_API_KEY"] = google_api_key
    print("GOOGLE_API_KEY loaded successfully.")

except Exception as e:
    print(f"Failed to load configuration: {e}")
    raise

GOOGLE_API_KEY loaded successfully.


### 2. Imports

In [16]:
from deepagents import create_deep_agent
from langgraph.graph import StateGraph
from typing import TypedDict, List, Dict, Any, Optional
from datetime import datetime

import time
# from pymongo import MongoClient
from functools import lru_cache
import logging
from pathlib import Path

import pandas as pd
import psycopg2
from logging.execution_logger import logger

ModuleNotFoundError: No module named 'pymongo'

### 3. State Definition

a Layered Deep-Agent Architecture:
app/
│
├── state/
│   ├── state_definition.py
│
├── memory/
│   ├── working_memory.py
│   ├── episodic_memory.py
│   ├── semantic_memory.py
│
├── database/
│   ├── mongodb_client.py
│   ├── postgres_client.py
│   ├── csv_loader.py
│
├── tools/
│   ├── mongo_tools.py
│   ├── sql_tools.py
│   ├── ml_tools.py
│   ├── report_tools.py
│
├── logging/
│   ├── execution_logger.py
│
├── agents/
│   ├── orchestrator_agent.py
│   ├── information_agent.py
│   ├── impact_agent.py
│   ├── report_agent.py
│   ├── evaluator_agent.py
│   ├── error_manager_agent.py
│
├── learning/
│   ├── reflection_engine.py
│   ├── reinforcement_manager.py
│
├── workflow/
│   ├── graph_definition.py
│
├── monitoring/
│   ├── metrics.py
│
├── tests/
│   ├── test_agents.py
│   ├── test_memory.py
│
└── main.py

In [17]:
   class WorkingMemory(TypedDict):
    retrieved_memories: List[Dict]
    active_context: List[Dict]
    supporting_evidence: List[Dict]
    related_events: List[Dict]


class EvaluationState(TypedDict):
    score: float
    strengths: List[str]
    weaknesses: List[str]
    recommendations: List[str]


class AgentState(TypedDict):

    # user request
    user_query: str

    # planning
    task_plan: List[str]

    # agent outputs
    market_event_data: List[Dict]

    churn_analysis: Dict

    impact_assessment: Dict

    report_draft: str

    evaluation: EvaluationState

    # memory
    working_memory: WorkingMemory

    # workflow
    current_step: str

    iteration_count: int

    error_log: List[str]

    final_response: str

### 4. Database Layer

In [ ]:
class MongoDBClient:

    def __init__(self, uri):

        self.client = MongoClient(uri)

        self.db = self.client["industry_db"]

        self.collection = self.db["events"]

    def search_events(self, keyword):

        return list(
            self.collection.find(
                {"keywords": keyword}
            )
        )


class ChurnDataLoader:

    def __init__(self, filepath):

        self.filepath = filepath

    def load(self):

        return pd.read_csv(
            self.filepath
        )

class MemoryDB:

    def __init__(self, config):

        self.conn = psycopg2.connect(**config)

    def save_reflection(
        self,
        reflection
    ):
        pass

    def retrieve_reflections(self):
        pass

### 5. Logging Layer

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format=(
        "%(asctime)s | "
        "%(name)s | "
        "%(levelname)s | "
        "%(message)s"
    )
)

logger = logging.getLogger(
    "deep_agent"
)

logger.info(
    "Impact Agent started"
)


### 6. Memory Layer

In [ ]:

class WorkingMemory:

    def update(
        self,
        state,
        new_context
    ):

        state["working_memory"][
            "active_context"
        ].append(new_context)

        return state

class EpisodicMemory:

    def store_episode(
        self,
        task,
        result
    ):
        pass

    def retrieve_similar(
        self,
        query
    ):
        pass

class SemanticMemory:

    def store_knowledge(
        self,
        knowledge
    ):
        pass

    def retrieve(
        self,
        query
    ):
        pass

### 7. Tool Layer

In [ ]:
class EventRetriever:

    def __init__(
        self,
        mongo_client
    ):
        self.mongo_client = mongo_client

    def retrieve_event(
        self,
        keyword
    ):

        return self.mongo_client.search_events(
            keyword
        )

class ChurnTool:

    def __init__(
        self,
        dataframe
    ):
        self.df = dataframe

    def churn_rate(self):

        return (
            self.df["Churn"] == 1
        ).mean()


from sklearn.ensemble import RandomForestClassifier


class ChurnImpactModel:

    def train(
        self,
        X,
        y
    ):

        model = RandomForestClassifier()

        model.fit(X, y)

        return model

### 8. Orchestrator Agent

In [ ]:
class OrchestratorAgent:

    def should_retry(
        self,
        state
    ):

        score = state[
            "evaluation"
        ]["score"]

        return score < 90


### 9. Information Collection Agent

In [ ]:
class InformationCollectionAgent:

    def __init__(
        self,
        retriever
    ):
        self.retriever = retriever

    def run(
        self,
        state
    ):

        logger.info(
            "Collecting event information"
        )

        events = self.retriever.retrieve_event(
            state["user_query"]
        )

        state[
            "market_event_data"
        ] = events

        return state

### 10. Impact Analysis Agent

In [ ]:
class ImpactAnalysisAgent:

    def __init__(
        self,
        churn_tool
    ):
        self.churn_tool = churn_tool

    def run(
        self,
        state
    ):

        logger.info(
            "Running impact analysis"
        )

        churn_rate = (
            self.churn_tool.churn_rate()
        )

        state[
            "impact_assessment"
        ] = {
            "estimated_churn_rate":
            churn_rate
        }

        return state

### 11. Report Writing Agent 

In [ ]:
class ReportAgent:

    def run(
        self,
        state
    ):

        report = f"""
# Competitor Analysis Report

## Event Summary

{state['market_event_data']}

## Impact

{state['impact_assessment']}
"""

        state[
            "report_draft"
        ] = report

        return state

### 12. Error Manager Agent

In [ ]:
class ErrorManagerAgent:

    def handle(
        self,
        state,
        error
    ):

        logger.error(str(error))

        state[
            "error_log"
        ].append(
            str(error)
        )

        return state

### 13. Evaluator Agent

In [ ]:
class EvaluatorAgent:

    def run(
        self,
        state
    ):

        score = 80

        state["evaluation"] = {

            "score": score,

            "strengths": [
                "Good structure"
            ],

            "weaknesses": [
                "Need more evidence"
            ],

            "recommendations": [
                "Retrieve more data"
            ]
        }

        return state

### 14. Learning layer

In [ ]:
class ReflectionEngine:

    def generate_reflection(
        self,
        state
    ):

        score = state[
            "evaluation"
        ]["score"]

        if score < 90:

            return {
                "lesson":
                "Need richer context"
            }

        return {
            "lesson":
            "Successful workflow"
        }
        
class ReinforcementManager:

    def update_policy(
        self,
        reflection
    ):

        print(
            "Policy updated:",
            reflection
        )

### 15. Workflow / Graph Definition

In [14]:
from langgraph.graph import (
    StateGraph,
    END
)

from state.state_definition import (
    AgentState
)


def build_graph(
    info_agent,
    impact_agent,
    report_agent,
    evaluator
):

    graph = StateGraph(
        AgentState
    )

    graph.add_node(
        "collect",
        info_agent.run
    )

    graph.add_node(
        "impact",
        impact_agent.run
    )

    graph.add_node(
        "report",
        report_agent.run
    )

    graph.add_node(
        "evaluate",
        evaluator.run
    )

    graph.set_entry_point(
        "collect"
    )

    graph.add_edge(
        "collect",
        "impact"
    )

    graph.add_edge(
        "impact",
        "report"
    )

    graph.add_edge(
        "report",
        "evaluate"
    )

    graph.add_edge(
        "evaluate",
        END
    )

    return graph.compile()

NameError: name 'StateGraph' is not defined

### 16. Monitoring Layer

In [ ]:
class MetricsTracker:

    def __init__(self):

        self.agent_calls = 0

        self.workflow_runs = 0

        self.failures = 0

    def record_call(self):

        self.agent_calls += 1

### 17. Main Execution

In [ ]:
from workflow.graph_definition import (
    build_graph
)

initial_state = {

    "user_query":
    "AT&T launches new plan",

    "task_plan": [],

    "market_event_data": [],

    "churn_analysis": {},

    "impact_assessment": {},

    "report_draft": "",

    "evaluation": {},

    "working_memory": {

        "retrieved_memories": [],

        "active_context": [],

        "supporting_evidence": [],

        "related_events": []
    },

    "current_step": "",

    "iteration_count": 0,

    "error_log": [],

    "final_response": ""
}

graph = build_graph(
    info_agent,
    impact_agent,
    report_agent,
    evaluator
)

result = graph.invoke(
    initial_state
)

print(
    result["report_draft"]
)